# 05 — Council Debate
**Political Council Project**

Run the full political debate between DEMOS and ARES.


## 1. Libraries


In [2]:


import os
import torch
import json

ADAPTER_PATH   = f'checkpoints/checkpoint-800'
VECTORDB_PATH  = f'data/vectordb'




## 2. Load Fine-tuned Model



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

print('Applying fine-tuned adapter...')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()  

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer.pad_token = tokenizer.eos_token

print('Model ready')
print(f'VRAM: {torch.cuda.memory_reserved(0)/1024**3:.1f} GB')


C:\Users\yassin\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading base model...


W0608 14:02:01.800000 3120 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   1%|          | 2/291 [00:00<01:17,  3.71it/s]C:\Users\yassin\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:15<00:00, 18.61it/s]


Applying fine-tuned adapter...
Model ready
VRAM: 4.0 GB


## 3. Load RAG Database


In [5]:
import chromadb
from sentence_transformers import SentenceTransformer

print('Loading ChromaDB...')
chroma_client = chromadb.PersistentClient(path=VECTORDB_PATH)
collection = chroma_client.get_collection('platforms')
print(f'Loaded {collection.count()} chunks')

print('Loading embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('RAG ready')


C:\Users\yassin\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.1.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


Loading ChromaDB...
Loaded 339 chunks
Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5944.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG ready


## 4. Core Functions

Three functions power the debate:

1. `retrieve_position` — gets official party stance from ChromaDB
2. `agent_speak` — generates the agent's debate response


In [ ]:
def retrieve_position(topic, ideology, n_results=2):

    query_vector = embedder.encode(topic).tolist()
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=n_results,
        where={'ideology': ideology}
    )
    chunks = results['documents'][0]
    return '\n'.join(chunks)


In [ ]:
def agent_speak(agent_name, ideology, topic, conversation_history, position):

    party = 'Democratic' if ideology == 'democrat' else 'Republican'


    history_text = ''
    if conversation_history:
        history_text = '\n'.join([
            f"{turn['agent']}: {turn['text']}"
            for turn in conversation_history
        ])
        history_text = f'\n\nDebate so far:\n{history_text}\n'

    if not conversation_history:
        instruction = f'Open the debate on {topic}. State your position clearly.'
    else:
        other = 'ARES' if agent_name == 'DEMOS' else 'DEMOS'
        instruction = f'Respond directly to {other}. Attack their argument. Be confrontational.'

    # Full prompt — fine-tuned voice + RAG position + conversation history
    prompt = (
        f'<s>[INST] You are {agent_name}, a {party} senator speaking about {topic}.\n\n'
        f'Your party\'s official position:\n{position}\n'
        f'{history_text}\n'
        f'{instruction} Keep response under 80 words. [/INST]'
    )

    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.85,      
            do_sample=True,
            repetition_penalty=1.1, 
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split('[/INST]')[-1].strip()
    return response


## 5. The Debate Loop



In [ ]:
def run_debate(topic, rounds=2):

    AGENTS = [
        {'name': 'DEMOS', 'ideology': 'democrat',   'icon': 'D'},
        {'name': 'ARES',  'ideology': 'republican', 'icon': 'R'},
    ]

    print('=' * 60)
    print(f'COUNCIL DEBATE: {topic.upper()}')
    print('=' * 60)
    print()


    positions = {}
    for agent in AGENTS:
        positions[agent['name']] = retrieve_position(topic, agent['ideology'])

    conversation_history = []


    for round_num in range(1, rounds + 1):
        print(f'--- Round {round_num} ---')
        print()

        for agent in AGENTS:
            name     = agent['name']
            ideology = agent['ideology']
            icon     = agent['icon']

            # Agent speaks
            response = agent_speak(
                name,
                ideology,
                topic,
                conversation_history,
                positions[name]
            )

            print(f'[{icon}] {name}:')
            print(response)
            print()

            # Add to history so next agent reads it
            conversation_history.append({
                'agent': name,
                'text':  response,
                'round': round_num
            })



    return conversation_history


## 6. Run The Debate


In [11]:
# Pick your topic
TOPIC = 'healthcare'

history= run_debate(topic=TOPIC, rounds=2)


COUNCIL DEBATE: HEALTHCARE

--- Round 1 ---

[D] DEMOS:
The Affordable Care Act improved access to health care for millions of Americans, but too many have been left behind. With the Build Back Better budget reconciliation bill we can extend ACA subsidies, expand Medicaid, and lower prescription drug costs.

[R] ARES:
Senator Debbie Stabnovos failed policies led to record-high inflation and rising health care costs across the country. Instead of continuing her failed economic policies, Congress should repeal every word of Obamacare and start over to deliver real health care solutions!

--- Round 2 ---

[D] DEMOS:
The ACA was supported by % of voters when it passed including % of you in Ohio (according to polls at the time). This isn't "Obamacare." It's a popular law that millions of Americans depend on to get health care. If you don't like it, propose something better.

[R] ARES:
I'm happy to take on any portion of Build Back Better and have debated it extensively - especially this bad

In [1]:
import importlib
import subprocess
import sys

libraries = [
    "datasets",
    "requests",
    "tqdm",
    "pandas",
    "transformers",
    "torch",
    "fitz",  # PyMuPDF
    "sentence_transformers",
    "chromadb",
    "peft",
    "trl",
    "bitsandbytes",
    "accelerate",
]

print(f"Python version: {sys.version}\n")
print(f"{'Library':<25} {'Version'}")
print("-" * 40)

for lib in libraries:
    try:
        module = importlib.import_module(lib)
        version = getattr(module, "__version__", None)
        if version is None:
            # fallback: pip show
            result = subprocess.run(
                [sys.executable, "-m", "pip", "show", lib],
                capture_output=True, text=True
            )
            for line in result.stdout.splitlines():
                if line.startswith("Version:"):
                    version = line.split(": ")[1]
                    break
        print(f"{lib:<25} {version or 'unknown'}")
    except ImportError:
        print(f"{lib:<25} NOT INSTALLED")

Python version: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]

Library                   Version
----------------------------------------


C:\Users\yassin\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\yassin\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.1.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


datasets                  4.8.5
requests                  2.32.4
tqdm                      4.67.1
pandas                    2.3.2
transformers              5.4.0
torch                     2.11.0+cu126
fitz                      1.27.2.3
sentence_transformers     5.5.1
chromadb                  1.5.9
peft                      0.19.1
trl                       NOT INSTALLED


W0608 23:02:00.969000 27068 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


bitsandbytes              0.49.2
accelerate                1.13.0
